In [ ]:
import requests
import pandas as pd

# Replace with your Discogs username and token
username = "<your_user_name>"
headers = {"Authorization": "Discogs token=<your_token>"}

url = f"https://api.discogs.com/users/{username}/collection/folders/0/releases"

all_releases = []
page = 1

while True:
    params = {"page": page, "per_page": 100}
    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    all_releases.extend(data["releases"])

    if page >= data["pagination"]["pages"]:
        break
    page += 1

# Convert to DataFrame
df_collection = pd.DataFrame(all_releases)
df_collection.head()

,id,instance_id,date_added,rating,basic_information,folder_id,notes
0,527808,420267904,2019-12-07T22:43:07-08:00,0,"{'id': 527808, 'master_id': 18006, 'master_url...",1,NaN
1,932348,2041477119,2025-07-15T05:26:05-07:00,0,"{'id': 932348, 'master_id': 91339, 'master_url...",1,NaN
2,797314,2041476917,2025-07-15T05:24:52-07:00,0,"{'id': 797314, 'master_id': 827828, 'master_ur...",1,NaN
3,948733,420261455,2019-12-07T22:11:06-08:00,0,"{'id': 948733, 'master_id': 268709, 'master_ur...",1,NaN
4,464491,422952523,2019-12-20T10:45:24-08:00,0,"{'id': 464491, 'master_id': 0, 'master_url': N...",1,NaN


In [21]:
df_collection=df_collection.drop(columns='id',axis=1)

In [22]:
"""
Normalize the nested JSON data in the 'basic_information' column and concatenate it with the original DataFrame to 
get separate columns for each attribute in the 'basic_information' dictionary. 
""" 
df_flat=pd.json_normalize(df_collection['basic_information'])
df_flat=pd.concat([df_collection.drop(columns=['basic_information']),df_flat],axis=1)


In [ ]:
def flatten_multiple_list_of_dicts(df, colnames):
    """
    Function to flatten multiple DataFrame columns that contain lists of dictionaries.
    Each column is expanded using the same logic as flatten_list_of_dicts.
    
    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the columns.
    colnames : list of str
        Column names to flatten.
    
    Returns
    -------
    pandas.DataFrame
        Original DataFrame with all specified columns flattened.
    """

    def expand_cell(cell, colname):
        if not isinstance(cell, list):
            return {}

        out = {}
        for i, d in enumerate(cell, start=1):
            for k, v in d.items():
                if isinstance(v, list):
                    # Split nested list sideways
                    for j, item in enumerate(v, start=1):
                        out[f"{colname}_{i}_{k}_{j}"] = item
                else:
                    out[f"{colname}_{i}_{k}"] = v
        return out

    df_out = df.copy()

    for col in colnames:
        expanded = df_out[col].apply(lambda x: expand_cell(x, col))
        expanded_df = pd.DataFrame(expanded.tolist())
        df_out = pd.concat([df_out.drop(columns=[col]), expanded_df], axis=1)

    return df_out


In [25]:
cols_to_flatten = ["artists","formats","labels"]
df = flatten_multiple_list_of_dicts(df_flat, cols_to_flatten)

In [26]:
# Keeping only the relevant columns for the final DataFrame
cols=['id','master_id','title','year','genres','styles','date_added','artists_1_name','artists_2_name','artists_1_id','artists_2_id','formats_1_name','formats_1_qty',
      'formats_1_descriptions_1','formats_1_descriptions_2','labels_1_name','labels_1_id']
df=df.loc[:,cols]

In [31]:
# Final genre and style expansion
df=pd.concat([df,df['genres'].apply(pd.Series).add_prefix('genre_').drop(columns=['genre_2','genre_3','genre_4']),
           df['styles'].apply(pd.Series).add_prefix('style_').drop(columns=['style_4','style_5','style_6','style_7','style_8','style_9','style_10'])],axis=1)

In [ ]:
# Removes the file if it exists and creates the data folder

from pathlib import Path
path=Path('data')
path.mkdir(parents=True, exist_ok=True)

# In case I need to delete
to_delete=Path('df_source.csv')
if to_delete.exists():
    to_delete.unlink()
    print(f"File deleted. {to_delete.name}")

In [ ]:
df["artists_2_id"] = df["artists_2_id"].astype(float).astype("Int64")
df["artists_2_id"] = df["artists_2_id"].fillna(0).astype(int)

In [34]:
df.to_csv('data/df_source.csv', index=False)